# 08. Lecke - Többügynökös tervezési minta


## Beállítás


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Miért többügynökös rendszerek?

A valós feladatok, mint például az utazástervezés, sokféle szaktudást igényelnek — logisztika, helyi ismeretek, költségvetés és még sok más. Egyetlen ügynök, aki mindent próbál kezelni, gyorsan nehézkessé válik.

A többügynökös rendszerek ezt **specializációval** oldják meg: minden ügynök egy adott területre koncentrál, így magasabb minőségű eredményeket produkál, mint egy általánosító. Emellett javítják a **skálázhatóságot** — új ügynököket adhatsz hozzá (pl. repülési szakértő, étteremkritikus), anélkül, hogy át kellene írni a meglévő munkafolyamatot. Az ügynökök egy strukturált folyamaton keresztül kapcsolódnak össze, átadva a kontextust egyik a másiknak.


## Speciális ügynökök létrehozása


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Szekvenciális munkafolyamat építése

A `WorkflowBuilder` lehetővé teszi, hogy az ügynököket irányított gráfba kössük össze. Itt létrehozunk egy egyszerű, kétlépéses folyamatot: a **TravelPlanner** megtervezi az útitervet, majd a **TravelConcierge** átnézi és továbbfejleszti azt.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## További ügynökök hozzáadása a munkafolyamathoz

A többügynökös minta egyik legnagyobb előnye, hogy milyen könnyen bővíthető. Az alábbiakban hozzáadunk egy **BudgetReviewer** ügynököt, amely ellenőrzi a tervet az utazó költségvetése alapján, jelöli azokat a tételeket, amelyek esetleg túlléphetik a költségkeretet, és pénzt megtakarító alternatívákat javasol. A munkafolyamat most három ügynököt futtat egymás után:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Hozz létre speciális ügynököket** — mindegyiknek egy fókuszált szereppel (tervezés, portaszolgálat, költségvetés felülvizsgálat).
2. **Kösd össze az ügynököket egy egymás után következő munkafolyamatba** a `WorkflowBuilder` és az `add_edge` használatával.
3. **Streameld a kimenetet** egy többrészes ügynök pipeline-ból, nyomon követve, hogy melyik ügynök beszél.
4. **Bővítsd a munkafolyamatot** új ügynökök hozzáadásával a lánchoz anélkül, hogy a meglévőket módosítanád.

A többrészes ügynök tervezési minta egyszerűvé teszi az egyes ügynököket, miközben gazdagabb, alaposabban átvizsgált eredményeket hoz létre, mint amit egyetlen ügynök önmagában elérhetne.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült fordítással. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítás hibákat vagy pontatlanságokat tartalmazhat. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén szakmai, emberi fordítást javaslunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból adódhat.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
